# Instruction tuning workbook: write the key lines yourself

Chapters 8–9 · Day 12 · **DGX Spark GPU first** · companion to
[03 · Instruction tuning end to end](03_instruction_tuning_end_to_end.ipynb)

Notebook 03 runs every step for you. This workbook asks you to **write the key lines**: usually one
to three, never a large block. Each exercise has the same rhythm:

1. **Read the task**, then write your lines where you see `...` in the exercise cell.
2. **Run the check.** It tests your code on small cases and prints ✅ or ❌ with a hint. Nothing
   crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then runs a small real
   experiment on Qwen3-0.6B.

Later steps use your code when its check passes, and the reference otherwise, so one stuck
exercise never blocks the rest.

| Part | You write |
|---|---|
| A · Data | template IDs, the answer span, the labels, the shifted loss |
| B · Patterns | a rank-1 matrix, the top-r approximation, the energy share, the LoRA parameter count |
| C · LoRA | the LoRA layer, freezing and wrapping Qwen3, one training step |
| D · QLoRA | block quantization, 4-bit packing, memory accounting |

**Environment:** the `Python (DGX Spark Native)` kernel with CUDA, and the Qwen3-0.6B tokenizer and
weights in the local Hugging Face cache (nothing is downloaded). Checks come from the tested module
`dongxi_llms.sft_workbook`.

In [ ]:
import gc, math, itertools, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from dongxi_llms import sft_workbook as wb

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this workbook is GPU first."
DEVICE, MODEL = "cuda", "Qwen/Qwen3-0.6B"
torch.manual_seed(0)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
tok = AutoTokenizer.from_pretrained(MODEL, local_files_only=True)
HEADER = tok.encode("<|im_start|>assistant\n", add_special_tokens=False)
EMPTY_THINK = tok.encode("<think>\n\n</think>\n\n", add_special_tokens=False)
IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def load_model(dtype=torch.bfloat16):
    return AutoModelForCausalLM.from_pretrained(MODEL, local_files_only=True, dtype=dtype).to(DEVICE)

def mine_or_ref(name, ok, reference):
    """Use your implementation if its check passed, otherwise the reference."""
    chosen = globals().get(name) if ok else reference
    print("using", "your" if ok else "the reference", "version of", name)
    return chosen

conversation = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "Give me one word for a very happy feeling."},
    {"role": "assistant", "content": "Elated."},
    {"role": "user",      "content": "Another one?"},
    {"role": "assistant", "content": "Joyful."},
]
print(torch.cuda.get_device_name(0), "| tokenizer ready")

---

# Part A · From a conversation to graded tokens

### Prediction

## Exercise 1 · token IDs from the chat template

Turn `conversation` into the token IDs the model reads, using Qwen3's **own** chat template.
One line. `tok.apply_chat_template(..., return_dict=True)` returns a dictionary; you want its
`"input_ids"` as a Python list.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
my_ids = ...   # one line

In [ ]:
ex1_ok = wb.check_render(globals().get("my_ids"), tok, conversation)

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
ids = list(tok.apply_chat_template(conversation, return_dict=True)["input_ids"])
wb.check_render(ids, tok, conversation, strict=True)
print(tok.decode(ids))

### Reference explanation

### Reference answer and mechanism

Never write the role markers by hand. The template adds `<|im_start|>`, role names,
`<|im_end|>` and, for Qwen3, an empty think block in the last assistant turn. Only the model's own
template shows exactly what training will see (learning note, topic 1).

### Prediction

## Exercise 2 · find one answer span

Given the position `i` where an assistant header `<|im_start|>assistant\n` starts, return
`(start, end)`, the graded span:

- `start` is the first token **after** the header (and after Qwen3's empty think block, if present)
- `end` is the position of the next `<|im_end|>`, **included** because the model must learn to stop

Three short lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_answer_span(ids, i):
    start = ...                                   # first token after the header
    if ids[start:start + len(EMPTY_THINK)] == EMPTY_THINK:
        start = ...                               # skip the empty think block
    end = ...                                     # index of the next IM_END at or after start
    return start, end

In [ ]:
ex2_ok = wb.check_answer_span(globals().get("my_answer_span"), tok)

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def answer_span_ref(ids, i):
    start = i + len(HEADER)
    if ids[start:start + len(EMPTY_THINK)] == EMPTY_THINK:
        start += len(EMPTY_THINK)
    return start, ids.index(IM_END, start)

wb.check_answer_span(answer_span_ref, tok, strict=True)
answer_span = mine_or_ref("my_answer_span", ex2_ok, answer_span_ref)

### Reference explanation

### Reference answer and mechanism

`list.index(value, start)` finds the next `<|im_end|>` at or after `start`. The empty think block
is skipped because, with thinking off, the inference prompt already contains it: the model never
has to generate it. Grade exactly what the model must produce.

### Prediction

## Exercise 3 · build the labels

Use `answer_span` (yours if it passed) to copy each graded span into `labels`. Everything else
stays `-100`. One line: slice assignment, end **included**.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_build_labels(ids):
    labels = [-100] * len(ids)
    for i in range(len(ids)):
        if ids[i:i + len(HEADER)] == HEADER:
            start, end = answer_span(ids, i)
            ...                                   # copy ids[start..end] into labels
    return labels

In [ ]:
ex3_ok = wb.check_build_labels(globals().get("my_build_labels"), tok)

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def build_labels_ref(ids):
    return wb.ref_build_labels(ids, HEADER, EMPTY_THINK, IM_END)

wb.check_build_labels(build_labels_ref, tok, strict=True)
build_labels = mine_or_ref("my_build_labels", ex3_ok, build_labels_ref)
labels = build_labels(ids)
print("graded:", [tok.decode([t]) for t, l in zip(ids, labels) if l != -100])

### Reference explanation

### Reference answer and mechanism

`end + 1` because Python slices exclude their end, and `<|im_end|>` must be graded. Labels are
the token IDs themselves, **unshifted**: the shift happens in the loss (Exercise 4).

### Prediction

## Exercise 4 · the shifted loss

Write the masked next-token loss for one sequence: `logits` has shape `[T, V]` and `labels`
has shape `[T]` (unshifted, `-100` where ungraded). Logit *t* predicts label *t + 1*. One line with
`F.cross_entropy`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_shifted_loss(logits, labels):
    return ...                                    # one line

In [ ]:
ex4_ok = wb.check_shifted_loss(globals().get("my_shifted_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def shifted_loss_ref(logits, labels):
    return F.cross_entropy(logits[:-1], labels[1:], ignore_index=-100)

wb.check_shifted_loss(shifted_loss_ref, strict=True)
shifted_loss = mine_or_ref("my_shifted_loss", ex4_ok, shifted_loss_ref)

model = load_model()                               # real check on Qwen3-0.6B
x, y = torch.tensor([ids], device=DEVICE), torch.tensor([labels], device=DEVICE)
with torch.no_grad():
    logits = model(x).logits.float()[0]
    hf = model(x, labels=y).loss.float()
print(f"your/reference loss {float(shifted_loss(logits, y[0])):.4f} | Hugging Face internal {float(hf):.4f}")

### Reference explanation

### Reference answer and mechanism

`logits[:-1]` drops the last prediction (nothing follows the end) and `labels[1:]` drops the
first label (nothing predicts it). `ignore_index=-100` removes ungraded positions from both the
sum and the count. Without the shift the shapes still match and you'd silently train a copying
task.

---

# Part B · Patterns: rank, approximation, and LoRA's budget

### Prediction

## Exercise 5 · one pattern

Build `[[1,2,3],[2,4,6],[3,6,9]]` as **one column × one row**. One line, using `torch.outer`
or `u[:, None] * v[None, :]`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
u = torch.tensor([1., 2., 3.])
v = torch.tensor([1., 2., 3.])
my_M = ...                                        # one pattern: column × row

In [ ]:
ex5_ok = wb.check_rank_one(globals().get("my_M"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
M = torch.outer(torch.tensor([1., 2., 3.]), torch.tensor([1., 2., 3.]))
wb.check_rank_one(M, strict=True)
M2 = M + torch.outer(torch.tensor([1., 0., -1.]), torch.tensor([0., 1., 0.]))
print("one pattern -> rank", int(torch.linalg.matrix_rank(M)), "| two patterns -> rank", int(torch.linalg.matrix_rank(M2)))

### Reference explanation

### Reference answer and mechanism

Each entry is `u[i] · v[j]`, so every row is the same shape scaled differently: one pattern,
rank 1. Adding a second, independent column × row pattern makes rank 2, like adding the promotion
to the sales table in our discussion.

### Prediction

## Exercise 6 · keep the r strongest patterns

Return the best rank-*r* approximation of `M`: keep the first `r` columns of `U`, the first `r`
values of `S` and the first `r` rows of `Vh`, then multiply back. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_top_r(M, r):
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)
    return ...                                    # one line

In [ ]:
ex6_ok = wb.check_top_r(globals().get("my_top_r"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
wb.check_top_r(wb.ref_top_r, strict=True)
top_r = mine_or_ref("my_top_r", ex6_ok, wb.ref_top_r)
img = plt.imread(plt.matplotlib.cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))
photo = torch.tensor(img[..., :3].mean(-1), dtype=torch.float32, device=DEVICE) / 255
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, r in zip(axes, [1, 5, 20, 50]):
    approx = top_r(photo, r)
    ax.imshow(approx.clamp(0, 1).cpu(), cmap="gray"); ax.set_axis_off()
    ax.set_title(f"r = {r}, error {float((approx - photo).norm() / photo.norm()):.0%}")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

`U[:, :r] * S[:r]` scales each kept column by its strength, then `@ Vh[:r]` adds up the `r`
column × row patterns. By the Eckart–Young theorem this is the best rank-*r* approximation. The
photo becomes recognizable around r = 20.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Your top-r function applied to a photograph](../figures/chapter-09/day-12-04_instruction_tuning_workbook-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

### Prediction

## Exercise 7 · how much of the strength is in the top r?

Return the share of a matrix's **energy** in its top `r` patterns: the sum of the top `r`
squared singular values divided by the sum of all squared singular values. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_energy(M, r):
    S = torch.linalg.svdvals(M)
    return ...                                    # one line

In [ ]:
ex7_ok = wb.check_energy(globals().get("my_energy"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def energy_ref(M, r):
    S = torch.linalg.svdvals(M)
    return float((S[:r] ** 2).sum() / (S ** 2).sum())

wb.check_energy(energy_ref, strict=True)
energy = mine_or_ref("my_energy", ex7_ok, energy_ref)
W = model.model.layers[10].self_attn.v_proj.weight.detach().float()
for name, A in (("photo", photo), ("pretrained v_proj", W), ("random matrix", torch.randn_like(W))):
    print(f"{name:<18} top-8 energy {energy(A, 8):6.1%} | top-64 {energy(A, 64):6.1%}")

### Reference explanation

### Reference answer and mechanism

Squared because energy adds up the squared sizes of the patterns, the same way the error of a
rank-*r* approximation is measured. The photo concentrates its energy; the pretrained weight
spreads it out much like a random matrix. That's why LoRA leaves `W` complete and only makes the
**update** low-rank.

### Prediction

## Exercise 8 · LoRA's parameter budget

How many trainable numbers does one LoRA adapter have, for a weight of shape `d_out × d_in` and
rank `r`? (`A` is `r × d_in`, `B` is `d_out × r`.) One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_lora_params(d_out, d_in, r):
    return ...                                    # one line

In [ ]:
ex8_ok = wb.check_lora_param_count(globals().get("my_lora_params"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def lora_params_ref(d_out, d_in, r):
    return r * d_in + d_out * r

wb.check_lora_param_count(lora_params_ref, strict=True)
q, v = model.model.layers[0].self_attn.q_proj.weight.shape, model.model.layers[0].self_attn.v_proj.weight.shape
per_layer = lora_params_ref(*q, 8) + lora_params_ref(*v, 8)
print(f"q_proj {tuple(q)}, v_proj {tuple(v)} -> {per_layer:,} per layer -> {28 * per_layer:,} for 28 layers")

### Reference explanation

### Reference answer and mechanism

`r · (d_in + d_out)` instead of `d_in · d_out`. For Qwen3's `q_proj` (2048 × 1024) at rank 8,
that's 24,576 numbers instead of 2,097,152, about 1.2%.

---

# Part C · LoRA on Qwen3-0.6B

### Prediction

## Exercise 9 · the LoRA layer

Complete the two key lines of a LoRA layer:

1. `B` starts at **zero**, shape `d_out × r`, on the same device as the base weight.
2. The forward pass returns the base output plus the scaled low-rank update `scale · x Aᵀ Bᵀ`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
class MyLoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.scale = base, alpha / r
        for p in self.base.parameters():
            p.requires_grad_(False)
        d_out, d_in = base.weight.shape
        device = base.weight.device
        self.A = nn.Parameter(torch.randn(r, d_in, device=device) / math.sqrt(d_in))
        self.B = ...                              # line 1: zeros, shape d_out × r

    def forward(self, x):
        return ...                                # line 2: base(x) + scaled low-rank update

In [ ]:
ex9_ok = wb.check_lora_linear(globals().get("MyLoRALinear"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
wb.check_lora_linear(wb.RefLoRALinear, strict=True)
LoRALinear = mine_or_ref("MyLoRALinear", ex9_ok, wb.RefLoRALinear)

### Reference explanation

### Reference answer and mechanism

`B = 0` makes the update exactly zero at step 0, so the wrapped model starts as the pretrained
one. `A` is random so that `B` receives a nonzero gradient straight away; if both were zero,
neither would ever move. `x @ A.T @ B.T` applies the `r` read → write patterns.

**Training data.** The same small authored set as notebook 03 (name the color, reverse two words),
rendered with Qwen3's template and masked with your `build_labels` (or the reference).

In [ ]:
colors = ["red", "blue", "green", "yellow", "purple", "orange", "black", "white"]
things = ["ball", "box", "kite", "cup", "hat", "car"]
words = ["sun", "moon", "tree", "river", "stone", "cloud"]
pairs = [(f"Reply with only the color: The {t} is {c}.", c) for c, t in itertools.product(colors, things)]
pairs += [(f"Reverse the two words: {a} {b}", f"{b} {a}") for a, b in itertools.permutations(words, 2)]

def make_batch(examples):
    enc = []
    for q, a in examples:
        ids_ = list(tok.apply_chat_template([{"role": "user", "content": q}, {"role": "assistant", "content": a}],
                                             return_dict=True)["input_ids"])
        enc.append((ids_, build_labels(ids_)))
    n = max(len(i) for i, _ in enc)
    batch = {"input_ids": torch.full((len(enc), n), tok.pad_token_id), "labels": torch.full((len(enc), n), -100),
             "attention_mask": torch.zeros((len(enc), n), dtype=torch.long)}
    for k, (i, l) in enumerate(enc):
        batch["input_ids"][k, :len(i)] = torch.tensor(i); batch["labels"][k, :len(l)] = torch.tensor(l)
        batch["attention_mask"][k, :len(i)] = 1
    return {key: value.to(DEVICE) for key, value in batch.items()}

TRAIN = make_batch(pairs)
del model; gc.collect(); torch.cuda.empty_cache()
print(len(pairs), "examples; graded tokens:", int((TRAIN["labels"] != -100).sum()))

### Prediction

## Exercise 10 · freeze Qwen3 and wrap q_proj and v_proj

On a fresh fp32 copy of Qwen3-0.6B:

1. freeze every pretrained parameter
2. wrap `q_proj` with `LoRALinear(..., r=8, alpha=16)`
3. wrap `v_proj` the same way
4. count the parameters that still require gradients

Four short lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
lora_model = load_model(torch.float32)
for p in lora_model.parameters():
    ...                                           # line 1: freeze
for layer in lora_model.model.layers:
    layer.self_attn.q_proj = ...                  # line 2: wrap q_proj
    layer.self_attn.v_proj = ...                  # line 3: wrap v_proj
my_trainable = ...                                # line 4: count trainable parameters

In [ ]:
ex10_ok = wb.check_trainable_count(globals().get("my_trainable"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
if not ex10_ok:                                   # rebuild with the reference steps
    lora_model = load_model(torch.float32)
    for p in lora_model.parameters():
        p.requires_grad_(False)
    for layer in lora_model.model.layers:
        layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj, r=8, alpha=16)
        layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj, r=8, alpha=16)
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
wb.check_trainable_count(trainable, strict=True)
total = sum(p.numel() for p in lora_model.parameters())
print(f"{trainable:,} of {total:,} parameters train ({trainable / total:.2%})")

### Reference explanation

### Reference answer and mechanism

Freeze **first**, then wrap: the new `A` and `B` are created after the freeze, so they're the
only parameters left trainable. Freezing after wrapping would freeze the adapters too.

### Prediction

## Exercise 11 · one training step

Write the four lines of one optimizer step: a forward pass with labels (the model returns
`.loss`), backward, an optimizer step, and clearing the gradients.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_train_step(model, batch, opt):
    loss = ...                                    # line 1: forward with labels -> .loss
    ...                                           # line 2: backward
    ...                                           # line 3: optimizer step
    ...                                           # line 4: clear gradients
    return float(loss)

In [ ]:
ex11_ok = wb.check_train_step(globals().get("my_train_step"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def train_step_ref(model, batch, opt):
    loss = model(**batch).loss
    loss.backward()
    opt.step()
    opt.zero_grad()
    return float(loss)

wb.check_train_step(train_step_ref, strict=True)
train_step = mine_or_ref("my_train_step", ex11_ok, train_step_ref)

base_q = lora_model.model.layers[10].self_attn.q_proj.base.weight.detach().clone()
opt = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=1e-3)
losses = [train_step(lora_model, TRAIN, opt) for _ in range(30)]
unchanged = torch.equal(base_q, lora_model.model.layers[10].self_attn.q_proj.base.weight)
print(f"loss {losses[0]:.3f} -> {losses[-1]:.2e} | frozen q_proj weight unchanged: {unchanged}")
plt.figure(figsize=(6, 3.2)); plt.plot(losses, color="#2563EB"); plt.yscale("log")
plt.xlabel("optimizer step"); plt.ylabel("training loss (graded tokens)"); plt.title("LoRA on Qwen3-0.6B")
plt.tight_layout(); plt.show()
assert unchanged and losses[-1] < losses[0]

### Reference explanation

### Reference answer and mechanism

Order matters: backward computes gradients, `step` uses them, `zero_grad` clears them so they
don't pile up into the next step. Only `A` and `B` move; the pretrained weights are bit-for-bit
unchanged.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![LoRA training loss using your training step](../figures/chapter-09/day-12-04_instruction_tuning_workbook-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part D · QLoRA's building blocks

In [ ]:
del lora_model, opt; gc.collect(); torch.cuda.empty_cache()
UNIFORM = torch.linspace(-1, 1, 16, device=DEVICE)
offset = 0.5 * (1 - 1 / 30 + 1 - 1 / 32)
pos = torch.special.ndtri(torch.linspace(offset, 0.5, 9, dtype=torch.float64)[:-1])
neg = -torch.special.ndtri(torch.linspace(offset, 0.5, 8, dtype=torch.float64)[:-1])
NF4 = torch.cat([pos, torch.zeros(1, dtype=torch.float64), neg]).sort().values
NF4 = (NF4 / NF4.abs().max()).float().to(DEVICE)
print("NF4 levels:", [round(v, 3) for v in NF4.tolist()])

### Prediction

## Exercise 12 · block-wise quantize and dequantize

Three lines:

1. `absmax`: the largest absolute weight in each block (keep the dimension)
2. `codes`: the index of the nearest level for each normalized weight `flat / absmax`
3. dequantize: level value × block scale, reshaped to the original shape

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_quantize(w, levels, block=64):
    flat = w.float().reshape(-1, block)
    absmax = ...                                  # line 1
    codes = ...                                   # line 2: argmin over |normalized - level|
    return codes, absmax.squeeze(1)

def my_dequantize(codes, absmax, levels, shape):
    return ...                                    # line 3

In [ ]:
ex12_ok = wb.check_quantize(globals().get("my_quantize"), globals().get("my_dequantize"), torch.linspace(-1, 1, 16))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
wb.check_quantize(wb.ref_quantize, wb.ref_dequantize, torch.linspace(-1, 1, 16), strict=True)
quantize = mine_or_ref("my_quantize", ex12_ok, wb.ref_quantize)
dequantize = mine_or_ref("my_dequantize", ex12_ok, wb.ref_dequantize)

W = load_model().model.layers[10].self_attn.q_proj.weight.detach().float()
for name, levels in (("uniform", UNIFORM), ("NF4", NF4)):
    for block in (W.numel(), 64):
        c, s = quantize(W, levels, block)
        err = float((dequantize(c, s, levels, W.shape) - W).norm() / W.norm())
        print(f"{name:<8} {'global scale' if block == W.numel() else 'blocks of 64':<13} relative error {err:.1%}")
gc.collect(); torch.cuda.empty_cache()

### Reference explanation

### Reference answer and mechanism

`unsqueeze(-1).sub(levels)` compares every weight with all 16 levels at once, and `argmin` picks
the nearest. On the real Qwen weight, blocks of 64 beat one global scale by a wide margin, and NF4
beats evenly spaced levels by a smaller one, matching Predicts 11–12 of notebook 03.

### Prediction

## Exercise 13 · pack two 4-bit codes into one byte

Codes are integers 0–15, so two fit in one byte. Pack the even-indexed code into the **high**
4 bits and the next one into the low 4 bits, then unpack. One line each.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_pack(codes):                                # codes: uint8, values 0..15, even count per row
    return ...                                    # (high << 4) | low

def my_unpack(packed):
    return ...                                    # high = byte >> 4, low = byte & 15, interleaved back

In [ ]:
ex13_ok = wb.check_pack(globals().get("my_pack"), globals().get("my_unpack"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def pack_ref(codes):
    return (codes[:, 0::2] << 4) | codes[:, 1::2]

def unpack_ref(packed):
    return torch.stack([packed >> 4, packed & 15], dim=-1).reshape(packed.shape[0], -1)

wb.check_pack(pack_ref, unpack_ref, strict=True)
codes, _ = wb.ref_quantize(W.cpu(), NF4.cpu())
packed = pack_ref(codes.to(torch.uint8))
print(f"{W.numel():,} weights: {codes.numel() * 2:,} bytes as bf16 -> {packed.numel():,} bytes packed")

### Reference explanation

### Reference answer and mechanism

`<< 4` shifts a code into the top half of the byte and `|` merges in the low half. `>> 4` and
`& 15` (binary `00001111`) take them apart again. That's the factor of 4 over bf16 for storage.

### Prediction

## Exercise 14 · memory accounting

Fill in the bytes per parameter for weights + gradients + AdamW state (adapters ignored):

- `full`: fp32 weight, fp32 gradient, two fp32 AdamW moments
- `lora`: a frozen bf16 weight only
- `qlora`: a frozen 4-bit weight only

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_memory_bytes(n_params, method):
    bytes_per_param = {"full": ..., "lora": ..., "qlora": ...}
    return n_params * bytes_per_param[method]

In [ ]:
ex14_ok = wb.check_memory(globals().get("my_memory_bytes"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def memory_bytes_ref(n_params, method):
    return n_params * {"full": 16, "lora": 2, "qlora": 0.5}[method]

wb.check_memory(memory_bytes_ref, strict=True)
for n, label in ((0.6e9, "0.6B"), (7e9, "7B"), (70e9, "70B")):
    print(label, " | ".join(f"{m} {memory_bytes_ref(n, m) / 2**30:7.1f} GiB" for m in ("full", "lora", "qlora")))

### Reference explanation

### Reference answer and mechanism

16 bytes = 4 (fp32 weight) + 4 (gradient) + 8 (two AdamW moments). LoRA keeps only the frozen
bf16 weight (2 bytes), QLoRA only 4 bits (0.5 bytes), plus small block scales in practice. Activations
come on top of all three.

## Reading the evidence

The checks test your code on small, known cases, and the reference cells then run it on real
Qwen3-0.6B weights. A ✅ means your lines behave like the reference on those cases. It isn't a
proof of correctness for every input. The training run is a short demo on easy authored tasks,
so it shows mechanics, not capability. For the full experiments (full fine-tuning vs. LoRA vs.
QLoRA, peak memory, the low-rank measurement), see
[notebook 03](03_instruction_tuning_end_to_end.ipynb).